In [2]:
!apt-get install -y nmap > /dev/null 2>&1
!pip install groq python-nmap requests tabulate colorama --quiet

import os, json, socket, ssl, warnings
import requests, nmap
from datetime import datetime
from tabulate import tabulate
from colorama import Fore, Style, init
from groq import Groq

warnings.filterwarnings("ignore")
init(autoreset=True)
print("All libraries ready ✓")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.9 MB/s eta 0:00:00
All libraries ready ✓


In [3]:
GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready")

Groq client ready


In [4]:
# ── Port Scanner ──────────────────────────────────────
def scan_ports(target, ports="21,22,23,25,53,80,443,3306,5432,8080,8443"):
    nm = nmap.PortScanner()
    print(f"  [*] Scanning ports on {target}...")
    try:
        nm.scan(hosts=target, ports=ports, arguments="-sV --open -T4")
        results = {"host": target, "open_ports": []}
        for host in nm.all_hosts():
            for proto in nm[host].all_protocols():
                for port in nm[host][proto].keys():
                    svc = nm[host][proto][port]
                    results["open_ports"].append({
                        "port": port, "state": svc["state"],
                        "service": svc["name"],
                        "version": svc.get("version", "unknown")
                    })
        return results
    except Exception as e:
        return {"host": target, "open_ports": [], "error": str(e)}

# ── HTTP Header Auditor ───────────────────────────────
SECURITY_HEADERS = [
    "Strict-Transport-Security", "Content-Security-Policy",
    "X-Frame-Options", "X-Content-Type-Options",
    "Referrer-Policy", "Permissions-Policy", "X-XSS-Protection"
]

def audit_headers(url):
    print(f"  [*] Auditing HTTP headers on {url}...")
    try:
        r = requests.get(url, timeout=10, verify=False,
                         headers={"User-Agent": "SecurityAuditor/1.0"})
        present, missing = [], []
        lower_keys = {k.lower() for k in r.headers}
        for h in SECURITY_HEADERS:
            if h.lower() in lower_keys:
                present.append({"header": h, "value": r.headers.get(h,"")})
            else:
                missing.append(h)
        return {"url": url, "status_code": r.status_code,
                "server": r.headers.get("Server", "not disclosed"),
                "present_headers": present, "missing_headers": missing}
    except Exception as e:
        return {"url": url, "error": str(e)}

# ── SSL/TLS Checker ───────────────────────────────────
def check_ssl(hostname, port=443):
    print(f"  [*] Checking SSL/TLS on {hostname}:{port}...")
    try:
        ctx = ssl.create_default_context()
        with ctx.wrap_socket(socket.socket(), server_hostname=hostname) as s:
            s.settimeout(10)
            s.connect((hostname, port))
            cert = s.getpeercert()
            proto = s.version()
        expiry = cert.get("notAfter", "")
        issues = []
        if expiry:
            exp_date = datetime.strptime(expiry, "%b %d %H:%M:%S %Y %Z")
            days_left = (exp_date - datetime.utcnow()).days
            if days_left < 30:
                issues.append(f"Cert expires in {days_left} days")
        if proto in ("TLSv1", "TLSv1.1", "SSLv3"):
            issues.append(f"Weak protocol: {proto}")
        return {"hostname": hostname, "protocol": proto,
                "expiry": expiry, "issues": issues}
    except Exception as e:
        return {"hostname": hostname, "error": str(e)}

print("Scanner modules loaded ✓")

Scanner modules loaded ✓


In [5]:
AUDIT_PROMPT = """You are a senior cybersecurity analyst.
Analyze these raw security scan results and return ONLY valid JSON.
No markdown, no explanation, just the JSON object.

For each finding include:
- title, category (port/header/ssl/config)
- severity: CRITICAL / HIGH / MEDIUM / LOW / INFO
- cvss: numeric 0.0-10.0
- description: 2-sentence plain-language risk explanation
- remediation: one specific actionable fix

Also include overall_risk and executive_summary.

Required format:
{
  "findings": [
    {
      "title": "...",
      "category": "...",
      "severity": "...",
      "cvss": 0.0,
      "description": "...",
      "remediation": "..."
    }
  ],
  "overall_risk": "CRITICAL|HIGH|MEDIUM|LOW",
  "executive_summary": "..."
}

RAW SCAN DATA:
"""

def analyze_with_ai(scan_data):
    print("  [*] Sending to Groq (Llama 3.3 70B) for analysis...")
    raw_json = json.dumps(scan_data, indent=2)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user",
                    "content": AUDIT_PROMPT + raw_json}],
        max_tokens=4096,
        temperature=0.1
    )

    text = response.choices[0].message.content.strip()
    # Strip markdown fences if model adds them
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:])
    if text.endswith("```"):
        text = "\n".join(text.split("\n")[:-1])
    return json.loads(text)

print("AI engine loaded ✓ (Llama 3.3 70B via Groq — FREE)")

AI engine loaded ✓ (Llama 3.3 70B via Groq — FREE)


In [6]:
SEVERITY_COLOR = {
    "CRITICAL": Fore.RED, "HIGH": Fore.YELLOW,
    "MEDIUM": Fore.CYAN, "LOW": Fore.GREEN, "INFO": Fore.WHITE
}

def print_report(analysis, target):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("\n" + "═"*65)
    print(f"  AI SECURITY AUDIT REPORT  |  {target}  |  {ts}")
    print("═"*65)

    risk = analysis.get("overall_risk", "UNKNOWN")
    c = SEVERITY_COLOR.get(risk, Fore.WHITE)
    print(f"\n  Overall Risk : {c}{risk}{Style.RESET_ALL}")
    print(f"\n  Summary: {analysis.get('executive_summary','N/A')}\n")

    findings = analysis.get("findings", [])
    rows = []
    for i, f in enumerate(findings, 1):
        sev = f.get("severity", "INFO")
        col = SEVERITY_COLOR.get(sev, "")
        rows.append([i, f.get("title","")[:42],
                     f"{col}{sev}{Style.RESET_ALL}",
                     f.get("cvss","N/A"),
                     f.get("category","").upper()])

    print(tabulate(rows,
        headers=["#","Finding","Severity","CVSS","Category"],
        tablefmt="rounded_outline"))

    print("\n  ── Detailed Findings ──")
    for i, f in enumerate(findings, 1):
        sev = f.get("severity","INFO")
        col = SEVERITY_COLOR.get(sev, Fore.WHITE)
        print(f"\n  [{i}] {col}{f.get('title','')}{Style.RESET_ALL}")
        print(f"      Severity : {col}{sev}{Style.RESET_ALL}  CVSS: {f.get('cvss','N/A')}")
        print(f"      Risk     : {f.get('description','')}")
        print(f"      Fix      : {Fore.GREEN}{f.get('remediation','')}{Style.RESET_ALL}")
    print("\n" + "═"*65)

def save_report(analysis, target):
    safe = target.replace(".","_").replace("/","_")
    fname = f"audit_{safe}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(fname, "w") as f:
        json.dump({"target": target, "timestamp": str(datetime.now()),
                   **analysis}, f, indent=2)
    print(f"  Report saved → {fname}")
    return fname

print("Report generator loaded ✓")

Report generator loaded ✓


In [7]:
# ── Set your target ───────────────────────────────────
# testphp.vulnweb.com is a legal intentionally-vulnerable site
TARGET_URL  = "http://testphp.vulnweb.com"
TARGET_HOST = "testphp.vulnweb.com"
# ─────────────────────────────────────────────────────

print(f"\n{'='*60}")
print(f" AI Security Audit → {TARGET_HOST}  (FREE via Groq)")
print(f"{'='*60}\n")

print("[1/3] Running scan modules...")
port_results   = scan_ports(TARGET_HOST)
header_results = audit_headers(TARGET_URL)
ssl_results    = check_ssl(TARGET_HOST)

raw_findings = {
    "port_scan":    port_results,
    "header_audit": header_results,
    "ssl_check":    ssl_results
}

print("\n[2/3] Sending to Groq AI for analysis...")
analysis = analyze_with_ai(raw_findings)

print("\n[3/3] Generating report...")
print_report(analysis, TARGET_HOST)
save_report(analysis, TARGET_HOST)


 AI Security Audit → testphp.vulnweb.com  (FREE via Groq)

[1/3] Running scan modules...
  [*] Scanning ports on testphp.vulnweb.com...
  [*] Auditing HTTP headers on http://testphp.vulnweb.com...
  [*] Checking SSL/TLS on testphp.vulnweb.com:443...

[2/3] Sending to Groq AI for analysis...
  [*] Sending to Groq (Llama 3.3 70B) for analysis...

[3/3] Generating report...

═════════════════════════════════════════════════════════════════
  AI SECURITY AUDIT REPORT  |  testphp.vulnweb.com  |  2026-05-17 21:04:18
═════════════════════════════════════════════════════════════════

  Overall Risk : INFO

  Summary: The scan results indicate that the host testphp.vulnweb.com is not responding to connection attempts, and the URL and SSL connections are timing out, likely due to a network or server issue.

╭─────┬────────────────────────┬────────────┬────────┬────────────╮
│   # │ Finding                │ Severity   │   CVSS │ Category   │
├─────┼────────────────────────┼────────────┼────────┼

'audit_testphp_vulnweb_com_20260517_210418.json'

In [8]:
# Analyze any custom findings without running a live scan
custom_findings = {
    "manual_review": {
        "issues": [
            "SQL queries built with string concatenation",
            "Passwords stored as MD5 (no salt)",
            "No rate limiting on /login endpoint",
            "Debug mode enabled in production",
            "AWS keys found in git commit history"
        ]
    }
}

analysis2 = analyze_with_ai(custom_findings)
print_report(analysis2, "manual-code-review")
save_report(analysis2, "manual-code-review")

  [*] Sending to Groq (Llama 3.3 70B) for analysis...

═════════════════════════════════════════════════════════════════
  AI SECURITY AUDIT REPORT  |  manual-code-review  |  2026-05-17 21:04:30
═════════════════════════════════════════════════════════════════

  Overall Risk : CRITICAL

  Summary: The application has multiple critical vulnerabilities, including SQL injection, insecure password storage, and exposed AWS keys, which could lead to significant security breaches and financial losses. Immediate remediation is necessary to address these vulnerabilities and prevent potential attacks.

╭─────┬──────────────────────────────────┬────────────┬────────┬────────────╮
│   # │ Finding                          │ Severity   │   CVSS │ Category   │
├─────┼──────────────────────────────────┼────────────┼────────┼────────────┤
│   1 │ SQL Injection Vulnerability      │ CRITICAL   │      9 │ CONFIG     │
│   2 │ Insecure Password Storage        │ HIGH       │      7 │ CONFIG     │
│   3 │ B

'audit_manual-code-review_20260517_210430.json'